Load source CSVs and write Bronze Delta tables.

In [0]:

# from pyspark.sql import functions as F
# from pyspark.sql.types import StructType

# SOURCE_PATH = "/Volumes/ecommerce/bronze/ecommerce/"

# datasets = {
#     "olist_orders_dataset.csv": "ecommerce.bronze.orders",
#     "olist_order_items_dataset.csv": "ecommerce.bronze.order_items",
#     "olist_order_payments_dataset.csv": "ecommerce.bronze.payments",
#     "olist_customers_dataset.csv": "ecommerce.bronze.customers",
#     "olist_geolocation_dataset.csv": "ecommerce.bronze.geolocation",
#     "olist_order_reviews_dataset.csv": "ecommerce.bronze.order_reviews",
#     "olist_products_dataset.csv": "ecommerce.bronze.products",
#     "olist_sellers_dataset.csv": "ecommerce.bronze.sellers",
# }

# for file_name, target_table in datasets.items():
#     file_path = SOURCE_PATH + file_name

#     existing = spark.table(target_table)

#     metadata_columns = {"ingestion_timestamp", "source_file"}
#     source_schema = StructType([
#         field for field in existing.schema.fields
#         if field.name not in metadata_columns
#     ])

#     new_data = (
#         spark.read
#         .format("csv")
#         .option("header", "true")
#         .schema(source_schema)
#         .load(file_path)
#         .withColumn("ingestion_timestamp", F.current_timestamp())
#         .withColumn("source_file", F.col("_metadata.file_path"))
#     )

#     # Compare using Spark's actual metadata path.
#     source_file_value = (
#         new_data.select("source_file").first()["source_file"]
#     )

#     already_loaded = (
#         existing
#         .filter(F.col("source_file") == source_file_value)
#         .limit(1)
#         .count() > 0
#     )

#     if already_loaded:
#         print(f"SKIPPED: {file_name} already loaded")
#         continue

#     row_count = new_data.count()

#     if row_count > 0:
#         (
#             new_data.write
#             .format("delta")
#             .mode("append")
#             .saveAsTable(target_table)
#         )

#     print(f"INGESTED: {file_name}, {row_count} rows")

# print("Incremental ingestion check completed.")


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StructType

SOURCE_PATH = "/Volumes/ecommerce/bronze/ecommerce/"
LEDGER_TABLE = "ecommerce.bronze.ingestion_ledger"

datasets = {
    "olist_orders_dataset.csv": "ecommerce.bronze.orders",
    "olist_order_items_dataset.csv": "ecommerce.bronze.order_items",
    "olist_order_payments_dataset.csv": "ecommerce.bronze.payments",
    "olist_customers_dataset.csv": "ecommerce.bronze.customers",
    "olist_geolocation_dataset.csv": "ecommerce.bronze.geolocation",
    "olist_order_reviews_dataset.csv": "ecommerce.bronze.order_reviews",
    "olist_products_dataset.csv": "ecommerce.bronze.products",
    "olist_sellers_dataset.csv": "ecommerce.bronze.sellers",
    # "test_orders_batch.csv": "ecommerce.bronze.orders",
}

for file_name, target_table in datasets.items():
    file_path = SOURCE_PATH + file_name
    print(f"\nChecking {file_name}")

    if not spark.catalog.tableExists(target_table):
        raise ValueError(
            f"Target table {target_table} does not exist. "
            "Run your initial Bronze setup first."
        )

    existing = spark.table(target_table)

    source_schema = StructType([
        field for field in existing.schema.fields
        if field.name not in {"ingestion_timestamp", "source_file"}
    ])

    new_data = (
        spark.read
        .format("csv")
        .option("header", "true")
        .schema(source_schema)
        .load(file_path)
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_path"))
    )

    actual_source_path = (
        new_data.select("source_file").first()["source_file"]
    )

    already_loaded = (
        spark.table(LEDGER_TABLE)
        .filter(
            (F.col("source_file") == actual_source_path)
            & (F.col("target_table") == target_table)
            & (F.col("status") == "SUCCESS")
        )
        .limit(1)
        .count() > 0
    )

    if already_loaded:
        print(f"SKIPPED: {file_name} already registered as successful")
        continue

    # Recovery: Bronze data may have been written even if the
    # corresponding ledger entry was not created successfully.
    existing_source_rows = (
        spark.table(target_table)
        .filter(F.col("source_file") == actual_source_path)
        .count()
    )

    if existing_source_rows > 0:
        print(
            f"RECONCILING: {file_name} already has "
            f"{existing_source_rows} rows in Bronze, but no successful "
            "ledger entry was found."
        )

        reconciliation_record = [(
            actual_source_path,
            target_table,
            "SUCCESS",
            existing_source_rows,
            None,
            f"Reconciled existing Bronze data for {file_name}"
        )]

        ledger_schema = """
            source_file STRING,
            target_table STRING,
            status STRING,
            records_processed BIGINT,
            ingested_at TIMESTAMP,
            details STRING
        """

        (
            spark.createDataFrame(
                reconciliation_record,
                schema=ledger_schema
            )
            .withColumn("ingested_at", F.current_timestamp())
            .write.format("delta")
            .mode("append")
            .saveAsTable(LEDGER_TABLE)
        )

        print(f"RECONCILED: {file_name}; duplicate ingestion avoided.")
        continue

    row_count = new_data.count()

    if row_count == 0:
        print(f"SKIPPED: {file_name} contains no data rows")
        continue

    # Append first. Only record SUCCESS if the append completes.
    (
        new_data.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )

    ledger_record = [(
        actual_source_path,
        target_table,
        "SUCCESS",
        row_count,
        None,
        f"Appended from {file_name}"
    )]

    ledger_schema = """
        source_file STRING,
        target_table STRING,
        status STRING,
        records_processed BIGINT,
        ingested_at TIMESTAMP,
        details STRING
    """

    (
        spark.createDataFrame(
            ledger_record, schema=ledger_schema
        )
        .withColumn("ingested_at", F.current_timestamp())
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(LEDGER_TABLE)
    )

    print(f"INGESTED: {row_count} rows from {file_name}")

print("\nLedger-based ingestion finished.")


validation

In [0]:
%sql
SELECT 'orders' AS table_name, COUNT(*) AS row_count
FROM ecommerce.bronze.orders

UNION ALL

SELECT 'order_items', COUNT(*)
FROM ecommerce.bronze.order_items

UNION ALL

SELECT 'payments', COUNT(*)
FROM ecommerce.bronze.payments

UNION ALL

SELECT 'customers', COUNT(*)
FROM ecommerce.bronze.customers

UNION ALL

SELECT 'geolocation', COUNT(*)
FROM ecommerce.bronze.geolocation;

In [0]:
%sql
SELECT
    source_file,
    target_table,
    status,
    records_processed,
    ingested_at
FROM ecommerce.bronze.ingestion_ledger
WHERE source_file LIKE '%test_orders_batch.csv%';